# Koddan Kasa · Levha 19 · 0,03 milimetre: tekrar oynatmak öğrenmek değil

Universal Robots'un föyü: **UR5e aynı noktaya ±0,03 mm farkla döner** (ISO 9283 pose repeatability, [resmî föy](https://www.universal-robots.com/manuals/latest/en/datasheets/ur5e/)).
Bu kadar iyi tekrarlamasının işe yaraması için fabrikada parçanın hep aynı yerde durması gerekir (kalıp / fikstür): kol bakmaz, kaydedileni oynatır. Buna **açık döngü** denir.

Bu defterde dünyayı kıpırdatıyoruz:
1. **Kayıt:** UR5e + Robotiq 2F-85 (MuJoCo Menagerie modelleri) 4 cm'lik bir küpü alıp kaldırıyor. Her 2 ms'de gönderilen 7 komut (6 eklem + tutucu) kaydediliyor: 2.350 adım.
2. **Replay:** aynı komut listesi, hiç değiştirilmeden; küp 0–5 cm, 8 yöne kaydırılmış. Kol küpe bakmıyor.
3. **Kural:** aynı zamanlama, ama hedef küpün **o anki yerinden** yeniden hesaplanıyor (ters kinematik). Öğrenme yok; küpün yeri simülasyondan okunuyor (kamera yok).

Başarı: hareket bitince küpün altı yerden ≥ 4 cm yukarıda **ve** küp iki parmağa da değiyor. Küp yalnız sürtünmeyle tutuluyor.

Videodaki sonuç (MuJoCo 3.17.0): replay **43/81** (2 cm'ye kadar 8/8, 3 cm 3/8, 4 cm 0/8), kural **81/81**.

## 1 · Kurulum (MuJoCo + yalnız UR5e ve 2F-85 modelleri)

In [ ]:
!pip -q install mujoco mediapy
# Birebir aynı sayılar için: !pip -q install mujoco==3.17.0 mediapy
![ -d mujoco_menagerie ] || git clone -q --depth 1 --filter=blob:none --sparse https://github.com/google-deepmind/mujoco_menagerie
!cd mujoco_menagerie && git sparse-checkout set universal_robots_ur5e robotiq_2f85 && ls

## 2 · Deney kodu

`src/v6/kk19v3/experiment.py` ile aynı. `build()` kolu, tutucuyu, zemini ve küpü kuruyor; `plan()` bir al-kaldır hareketini her 2 ms için komut listesine çeviriyor; `run()` bir komut listesini adım adım oynatıp sonucu ölçüyor.

In [ ]:
import math, os, sys
from pathlib import Path
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")  # Colab'da ekransız çizim
import mujoco
import numpy as np
import warnings
warnings.filterwarnings("ignore", message="Attach conflict")  # tutucunun kendi temas ayarları ayrıca veriliyor (build)

MEN = Path(os.environ.get("MENAGERIE", "mujoco_menagerie"))

DT = 0.002
CUBE_HALF = 0.02  # 4 cm cube
CUBE_MASS = 0.1
CUBE_HOME = np.array([0.0, 0.5])  # recording spot: 50 cm in front of the base (the arm's "front" is +y here)
OFFSETS_CM = [0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5]
DIRS_DEG = [0, 45, 90, 135, 180, 225, 270, 315]  # 0 = +x (along the closing direction of the jaws), 90 = +y (away from base)
ARM = ["shoulder_pan_joint", "shoulder_lift_joint", "elbow_joint", "wrist_1_joint", "wrist_2_joint", "wrist_3_joint"]
ACT = ["shoulder_pan", "shoulder_lift", "elbow", "wrist_1", "wrist_2", "wrist_3"]
G_OPEN, G_CLOSE = 0.0, 255.0  # Robotiq 2F-85 command (0 = fully open, 85 mm)
PRE_M, LIFT_M, SUCCESS_LIFT_M = 0.10, 0.12, 0.04
HOME_Q = np.array([-1.5708, -1.5708, 1.5708, -1.5708, -1.5708, 0.0])  # Menagerie "home" keyframe: tool pointing down

T = dict(settle=0.3, pre=1.5, down=2.3, close=3.1, lift=4.1, hold=4.7)


def build(cube_xy):
    arm = mujoco.MjSpec.from_file(str(MEN / "universal_robots_ur5e/ur5e.xml"))
    grip = mujoco.MjSpec.from_file(str(MEN / "robotiq_2f85/2f85.xml"))
    arm.attach(grip, prefix="g_", site=arm.site("attachment_site"))
    arm.option.timestep = DT
    arm.option.impratio = 10  # the 2F-85 file's own contact settings (attach keeps the parent's)
    arm.option.cone = mujoco.mjtCone.mjCONE_ELLIPTIC
    wb = arm.worldbody
    wb.add_geom(name="floor", type=mujoco.mjtGeom.mjGEOM_PLANE, size=[2, 2, 0.05], rgba=[0.3, 0.3, 0.3, 1])
    b = wb.add_body(name="cube", pos=[cube_xy[0], cube_xy[1], CUBE_HALF])
    b.add_freejoint()
    b.add_geom(name="cube_geom", type=mujoco.mjtGeom.mjGEOM_BOX, size=[CUBE_HALF] * 3, mass=CUBE_MASS,
               friction=[1.0, 0.01, 0.001], condim=4, rgba=[0.86, 0.12, 0.09, 1])
    m = arm.compile()
    return arm, m, mujoco.MjData(m)


def ids(m):
    pads = {"left": set(), "right": set()}
    for g in range(m.ngeom):
        if m.geom_contype[g] == 0 and m.geom_conaffinity[g] == 0:
            continue
        body = mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_BODY, m.geom_bodyid[g]) or ""
        for side in pads:
            if body.startswith(f"g_{side}_") and ("pad" in body or "follower" in body):
                pads[side].add(g)
    return dict(cube=m.geom("cube_geom").id, cube_body=m.body("cube").id, site=m.site("g_pinch").id,
                qadr=[m.joint(j).qposadr[0] for j in ARM], dadr=[m.joint(j).dofadr[0] for j in ARM],
                act=[m.actuator(a).id for a in ACT], grip=m.actuator("g_fingers_actuator").id, pads=pads)


def ik(m, I, target, q0, iters=2000):
    """Damped least squares on the 6 arm joints: pinch site at `target`, approach (site z) straight down,
    jaws closing along world x (site y = +x). Kinematics only."""
    dd = mujoco.MjData(m)
    dd.qpos[I["qadr"]] = q0
    jp, jr = np.zeros((3, m.nv)), np.zeros((3, m.nv))
    za, ya = np.array([0, 0, -1.0]), np.array([1.0, 0, 0])
    for _ in range(iters):
        mujoco.mj_kinematics(m, dd)
        mujoco.mj_comPos(m, dd)
        R = dd.site_xmat[I["site"]].reshape(3, 3)
        ep = np.asarray(target) - dd.site_xpos[I["site"]]
        er = 0.5 * (np.cross(R[:, 2], za) + np.cross(R[:, 1], ya))
        if np.linalg.norm(ep) < 1e-4 and np.linalg.norm(er) < 1e-3:
            break
        mujoco.mj_jacSite(m, dd, jp, jr, I["site"])
        J = np.vstack([jp, jr])[:, I["dadr"]]
        dq = J.T @ np.linalg.solve(J @ J.T + 1e-4 * np.eye(6), np.concatenate([ep, er]))
        for k, a in enumerate(I["qadr"]):
            dd.qpos[a] += 0.7 * dq[k]
    mujoco.mj_kinematics(m, dd)
    return np.array([dd.qpos[a] for a in I["qadr"]]), float(np.linalg.norm(np.asarray(target) - dd.site_xpos[I["site"]]))


def minjerk(u):
    u = min(1.0, max(0.0, u))
    return u ** 3 * (10 - 15 * u + 6 * u * u)


def plan(m, I, cube_xy):
    """Joint waypoints for a top-down pick at cube_xy → the command for every physics step."""
    zc = CUBE_HALF
    q_down, e2 = ik(m, I, [cube_xy[0], cube_xy[1], zc - 0.005], HOME_Q)  # pads straddle the cube's middle
    q_pre, e1 = ik(m, I, [cube_xy[0], cube_xy[1], zc + PRE_M], q_down)
    q_lift, e3 = ik(m, I, [cube_xy[0], cube_xy[1], zc + LIFT_M], q_down)
    home = HOME_Q
    n = int(round(T["hold"] / DT))
    ctrl = np.zeros((n, 7))
    for i in range(n):
        t = (i + 1) * DT
        if t < T["settle"]:
            q, g = home, G_OPEN
        elif t < T["pre"]:
            q, g = home + (q_pre - home) * minjerk((t - T["settle"]) / (T["pre"] - T["settle"])), G_OPEN
        elif t < T["down"]:
            q, g = q_pre + (q_down - q_pre) * minjerk((t - T["pre"]) / (T["down"] - T["pre"])), G_OPEN
        elif t < T["close"]:
            q, g = q_down, G_OPEN + (G_CLOSE - G_OPEN) * minjerk((t - T["down"]) / (T["close"] - T["down"]))
        elif t < T["lift"]:
            q, g = q_down + (q_lift - q_down) * minjerk((t - T["close"]) / (T["lift"] - T["close"])), G_CLOSE
        else:
            q, g = q_lift, G_CLOSE
        ctrl[i, :6] = q
        ctrl[i, 6] = g
    return ctrl, max(e1, e2, e3)


def touching(m, d, I):
    side = {"left": False, "right": False}
    for k in range(d.ncon):
        c = d.contact[k]
        if I["cube"] in (c.geom1, c.geom2):
            o = c.geom2 if c.geom1 == I["cube"] else c.geom1
            for s in side:
                side[s] |= o in I["pads"][s]
    return side


def run(cube_xy, ctrl, keep=False):
    _, m, d = build(cube_xy)
    I = ids(m)
    d.qpos[I["qadr"]] = HOME_Q
    d.ctrl[I["act"]] = HOME_Q
    mujoco.mj_forward(m, d)
    start = d.xpos[I["cube_body"]].copy()
    first_touch, qs, tip = None, [], []
    for i, c in enumerate(ctrl):
        d.ctrl[I["act"]] = c[:6]
        d.ctrl[I["grip"]] = c[6]
        mujoco.mj_step(m, d)
        if first_touch is None:
            s = touching(m, d, I)
            if s["left"] or s["right"]:
                first_touch = round((i + 1) * DT, 3)
        if keep:
            qs.append(d.qpos.copy())
        if i % 5 == 0:
            tip.append(d.site_xpos[I["site"]].copy())
    s = touching(m, d, I)
    end = d.xpos[I["cube_body"]].copy()
    bottom = end[2] - CUBE_HALF
    lifted = bottom >= SUCCESS_LIFT_M
    moved = float(np.linalg.norm(end[:2] - start[:2]))
    ok = bool(lifted and s["left"] and s["right"])
    if ok:
        kind = "tuttu"
    elif first_touch is None:
        kind = "ıskaladı"
    elif not lifted:
        kind = "itti"
    else:
        kind = "tek çene"
    rec = dict(success=ok, outcome=kind, lift_cm=round(bottom * 100, 2), moved_cm=round(moved * 100, 2),
               pad_left=bool(s["left"]), pad_right=bool(s["right"]), first_touch_s=first_touch)
    return rec, (np.array(qs) if keep else None), np.array(tip)


def wilson(k, n, z=1.96):
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (round(max(0, c - h), 3), round(min(1, c + h), 3))


def xy_of(r, a):
    return CUBE_HOME + r / 100 * np.array([math.cos(math.radians(a)), math.sin(math.radians(a))])


def sweep(rec_ctrl, mode, keep_pairs):
    rows, traces = [], {}
    for r in OFFSETS_CM:
        for a in DIRS_DEG if r > 0 else [0]:
            xy = xy_of(r, a)
            if mode == "replay":
                ctrl, ik_err = rec_ctrl, None
            else:
                _, m0, _ = build(xy)
                ctrl, ik_err = plan(m0, ids(m0), xy)
            keep = (r, a) in keep_pairs
            res, qs, _ = run(xy, ctrl, keep=keep)
            res.update(offset_cm=r, dir_deg=a, cube_xy=[round(float(v), 4) for v in xy])
            if ik_err is not None:
                res["ik_err_mm"] = round(ik_err * 1000, 2)
            rows.append(res)
            if keep:
                traces[(r, a)] = (qs, ctrl)
            print(f"{mode:7s} {r:3.1f} cm {a:3d}° → {res['outcome']:9s} lift {res['lift_cm']:5.2f} cm moved {res['moved_cm']:5.2f}", flush=True)
    return rows, traces


def summarize(rows):
    out = []
    for r in OFFSETS_CM:
        sel = [x for x in rows if x["offset_cm"] == r]
        k = sum(x["success"] for x in sel)
        out.append(dict(offset_cm=r, n=len(sel), success=k, rate=round(k / len(sel), 3), wilson95=wilson(k, len(sel)),
                        outcomes={o: sum(x["outcome"] == o for x in sel) for o in sorted({x["outcome"] for x in sel})}))
    return out


def _repeat_check(rec_ctrl, n=5):
    """Replays the recording n times with the cube where it was recorded: tool path spread (sim is deterministic)."""
    tips = [run(CUBE_HOME, rec_ctrl)[2] for _ in range(n)]
    return round(float(max(np.abs(t - tips[0]).max() for t in tips) * 1000), 6)

## 3 · Kayıt

Küp kayıt yerindeyken tek bir hareket planlanıyor. Bu liste artık **ezber**: replay'de hiç değişmeyecek.

In [ ]:
_, m0, _ = build(CUBE_HOME)
I0 = ids(m0)
KAYIT, ik_err = plan(m0, I0, CUBE_HOME)
print("kayıt:", KAYIT.shape[0], "adım ×", KAYIT.shape[1], "komut   (IK hatası", round(ik_err * 1000, 3), "mm)")
print("1.200. adımın komutları:", np.round(KAYIT[1199], 2))
print("kayıt yerinde:", run(CUBE_HOME, KAYIT)[0])

## 4 · ✏️ BURAYI DEĞİŞTİR

Küpü kaydır: `KAYDIR_CM` (0–5) ve `YON_DERECE` (0 = +x, parmakların kapandığı yön; 90 = +y, tabandan uzağa; 180, 270 …).
`MOD = "replay"` aynı ezberi oynatır, `MOD = "kural"` küpün yerine bakıp yeniden nişan alır.

In [ ]:
# ===================== BURAYI DEĞİŞTİR =====================
KAYDIR_CM = 3      # küp kayıttaki yerinden kaç cm kaysın
YON_DERECE = 90    # hangi yöne
MOD = "replay"     # "replay" ya da "kural"
# ===========================================================

xy = xy_of(KAYDIR_CM, YON_DERECE)
if MOD == "replay":
    komut = KAYIT
else:
    _, mk, _ = build(xy)
    komut, _ = plan(mk, ids(mk), xy)
sonuc, _, _ = run(xy, komut)
print(f"küp {KAYDIR_CM} cm, {YON_DERECE}° · {MOD}: {sonuc['outcome']}  (küpün altı yerden {sonuc['lift_cm']:.1f} cm, yerinden {sonuc['moved_cm']:.1f} cm kaydı)")

### Videosu (480×480, 30 fps)

In [ ]:
import mediapy as media

def video(xy, komut, size=480, fps=30):
    _, m, d = build(xy)
    I = ids(m)
    d.qpos[I["qadr"]] = HOME_Q
    d.ctrl[I["act"]] = HOME_Q
    mujoco.mj_forward(m, d)
    r = mujoco.Renderer(m, height=size, width=size)
    cam = mujoco.MjvCamera()
    cam.lookat[:] = [CUBE_HOME[0], CUBE_HOME[1], 0.06]
    cam.distance, cam.azimuth, cam.elevation = 0.7, 262, -28
    frames, every = [], int(round(1 / fps / DT))
    for i, c in enumerate(komut):
        d.ctrl[I["act"]] = c[:6]
        d.ctrl[I["grip"]] = c[6]
        mujoco.mj_step(m, d)
        if i % every == 0:
            r.update_scene(d, camera=cam)
            frames.append(r.render())
    r.close()
    return frames

media.show_video(video(xy, komut), fps=30)

## 5 · Tüm tarama: 0–5 cm × 8 yön = 81 deneme, iki mod

Her satırı yeniden koşup bizim sayılarla karşılaştırıyor (`public/v6/kk-19-v3/data/replay.json`, MuJoCo 3.17.0). Farklı bir MuJoCo ya da Menagerie sürümünde sınırdaki birkaç deneme değişebilir.

In [ ]:
OURS = {"replay": [1, 8, 8, 8, 8, 5, 3, 2, 0, 0, 0], "kural": [1] + [8] * 10}

tablo = {}
for mod in ("replay", "kural"):
    tablo[mod] = []
    for r in OFFSETS_CM:
        k = n = 0
        for a in (DIRS_DEG if r > 0 else [0]):
            xy = xy_of(r, a)
            if mod == "replay":
                komut = KAYIT
            else:
                _, mk, _ = build(xy)
                komut, _ = plan(mk, ids(mk), xy)
            k += run(xy, komut)[0]["success"]
            n += 1
        tablo[mod].append(k)
print(f"{'kaydırma':>9} {'replay':>8} {'kural':>7}")
for i, r in enumerate(OFFSETS_CM):
    n = 1 if r == 0 else 8
    print(f"{r:>7} cm {tablo['replay'][i]:>5}/{n} {tablo['kural'][i]:>5}/{n}")
tr, tk = sum(tablo["replay"]), sum(tablo["kural"])
print(f"\ntoplam: replay {tr}/81, kural {tk}/81   (videoda: 43/81, 81/81)")
print("bizimkiyle aynı mı:", tablo["replay"] == OURS["replay"] and tablo["kural"] == OURS["kural"])

## 6 · Soru

Kural küpün **yerine** baktı ve 81'de 81 tuttu; hiçbir şey öğrenmedi. Peki mutfakta neden yetmez?
İpucu: bu deneyde değişen tek şey konumdu. Küpün yerine bir bardak, bir domates, kapağı kapalı bir kavanoz koyduğunu düşün: `hedef = küpün yeri` satırı neyi bilmiyor? Tahminin videonun yorumlarına.